## Импорт и определение библиотек

In [1]:
# отключение параллелизма
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Установка базовых пакетов + новые для улучшений
!pip install langchain langchain-community langchain_huggingface unstructured pypandoc -q
!pip install faiss-cpu -q
!pip install langchain-ollama -q
!pip install markdown
!pip install cohere -q  # Для реранкинга
!pip install openai -q  # Для OpenRouter

In [6]:
# импортируем библиотеки
from langchain.document_loaders import WebBaseLoader, PyPDFLoader, UnstructuredMarkdownLoader
from langchain_core.documents import Document
from langchain.prompts import PromptTemplate
from langchain.tools import Tool
from langchain.agents import initialize_agent
from langchain.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings
import tqdm as notebook_tqdm
from langchain_community.vectorstores import FAISS
from langchain.llms import Ollama
from langchain_ollama import OllamaLLM
from langchain.text_splitter import TextSplitter
from typing import List
import re
import json
from pathlib import Path
import numpy as np
import logging
from collections import Counter

# Настройка логирования
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# Готовим коктейль с помощью RAG
## Чутка `embeddings` с `HuggingFace` для оснастки

In [4]:
# Чтение токена из файла
def read_token(file_path): 
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith("HUGGINGFACEHUB_API_TOKEN"):
                return line.split('=')[1].strip()
    raise ValueError("Token not found in the specified file.")

# Загрузка токена
hf_token = read_token('../.env')

# Формирование эмбеддингов
hf_embeddings_model = HuggingFaceEmbeddings(
    model_name="cointegrated/LaBSE-en-ru",  
    #"sentence-transformers/paraphrase-multilingual-mpnet-base-v2", 
    model_kwargs={"device": "cpu", "token": hf_token},
    encode_kwargs={
        'normalize_embeddings': True  # Для косинусной схожести
    }
)

/home/elijah/Documents/LLM/cocktail_qr/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Можно пропустить (если используете наполненную чашу векторного хранилища)
### Немного `Document Loader`

In [13]:
# Загрузка Markdown файла от бармена-литературоведа
loader = UnstructuredMarkdownLoader('../docs/Federle_pandoc.md', mode="elements")
doc_markdown_mistral = loader.load()

# Объединяем все элементы в один текст
full_text = "\n".join([doc.page_content for doc in doc_markdown_mistral])
print(f'Общий размер текста: {len(full_text)} символов')

# Создаем один Document объект для дальнейшей обработки
doc_combined = [Document(page_content=full_text)]

Общий размер текста: 114655 символов


### Исполним `Text splitters` и разобьём на чанки

In [14]:
class ImprovedRecipeSplitter(TextSplitter):
    """
    Улучшенный сплиттер для рецепт-ориентированного чанкования с метаданными
    """
    
    def __init__(self):
        super().__init__(chunk_size=2000, chunk_overlap=200)
    
    def extract_ingredients(self, text: str) -> List[str]:
        """Извлекает список ингредиентов из текста рецепта"""
        ingredients = []
        
        # Расширенные паттерны для поиска ингредиентов
        ingredient_patterns = [
            r'(\d+\s*мл\s+[^;\n]+)',
            r'(\d+\s*г\s+[^;\n]+)', 
            r'(\d+\s*столов[а-я]*\s*ложк[а-я]*\s+[^;\n]+)',
            r'(\d+\s*чайн[а-я]*\s*ложк[а-я]*\s+[^;\n]+)',
            r'(\d+\s*банк[а-я]*\s*[^;\n]+)',
            r'(\d+\s*кубик[а-я]*\s*[^;\n]+)',
            r'(\d+\s*долек?\s*[^;\n]+)',
            # Ключевые алкогольные напитки
            r'([^.\n]*(?:водк|виск|джин|ром|текил|ликер|вин|коньяк|абсент)[а-я]*[^.\n]*)',
            # Соки и безалкогольные
            r'([^.\n]*(?:сок|лимонад|газировк|тоник|содов|вод)[а-я]*[^.\n]*)',
        ]
        
        for pattern in ingredient_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            ingredients.extend(matches)
        
        # Очистка и нормализация
        cleaned_ingredients = []
        for ingredient in ingredients:
            clean_ing = re.sub(r'[^\w\s]', '', ingredient.lower().strip())
            if len(clean_ing) > 3:
                cleaned_ingredients.append(clean_ing)
        
        return list(set(cleaned_ingredients))
    
    def extract_recipe_name(self, text: str) -> str:
        """Извлекает название рецепта"""
        lines = text.split('\n')
        for i, line in enumerate(lines[:5]):
            line = line.strip()
            if line and len(line) < 100:
                # Исключаем литературные справки
                if not any(word in line.lower() for word in ['роман', 'книга', 'автор', 'писатель', 'произведение']):
                    return line
        return "Неизвестный рецепт"
    
    def clean_text(self, text: str) -> str:
        """Очищает текст от разметки"""
        # Убираем Markdown разметку
        text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
        text = re.sub(r'\*([^*]+)\*', r'\1', text) 
        text = re.sub(r'`([^`]+)`', r'\1', text)
        text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
        
        # Нормализуем пробелы
        text = re.sub(r'\n\s*\n', '\n\n', text)
        text = re.sub(r' +', ' ', text)
        
        return text.strip()
    
    def split_text(self, text: str) -> List[str]:
        """Разбивает на рецепты с сохранением целостности"""
        cleaned_text = self.clean_text(text)
        chunks = []
        current_chunk = []
        
        lines = cleaned_text.split('\n')
        i = 0
        
        while i < len(lines):
            line = lines[i].strip()
            
            # Определяем начало нового рецепта
            is_new_recipe = (
                i + 1 < len(lines) and 
                line and 
                len(line) < 100 and
                lines[i+1].strip().isupper() and 
                '(' in lines[i+1] and ')' in lines[i+1]
            )
            
            if is_new_recipe and current_chunk:
                chunk_text = '\n'.join(current_chunk)
                if len(chunk_text.strip()) > 100:
                    chunks.append(chunk_text)
                current_chunk = []
            
            current_chunk.append(line)
            i += 1
        
        # Последний чанк
        if current_chunk:
            chunk_text = '\n'.join(current_chunk)
            if len(chunk_text.strip()) > 100:
                chunks.append(chunk_text)
        
        return chunks
    
    def create_documents_with_metadata(self, texts: List[str]) -> List[Document]:
        """Создает документы с метаданными"""
        documents = []
        
        for i, text in enumerate(texts):
            recipe_name = self.extract_recipe_name(text)
            ingredients = self.extract_ingredients(text)
            
            # Определяем тип напитка
            drink_type = "коктейль"
            if any(word in text.lower() for word in ['пунш', 'глинтвейн', 'грог']):
                drink_type = "горячий напиток"
            elif any(word in text.lower() for word in ['настойка', 'ликер']):
                drink_type = "настойка"
            elif any(word in text.lower() for word in ['безалкогольный', 'сок', 'лимонад']):
                drink_type = "безалкогольный"
            
            doc = Document(
                page_content=text,
                metadata={
                    'recipe_name': recipe_name,
                    'ingredients': ingredients,
                    'ingredients_str': ', '.join(ingredients),
                    'drink_type': drink_type,
                    'chunk_id': i,
                    'char_count': len(text)
                }
            )
            documents.append(doc)
        
        return documents

# Инициализируем улучшенный сплиттер
improved_splitter = ImprovedRecipeSplitter()


In [18]:
# Разбиваем текст на чанки с улучшенным алгоритмом
chunks_text = improved_splitter.split_text(full_text)
print(f'Количество текстовых чанков: {len(chunks_text)}')

# Создаем документы с богатыми метаданными
chunks_with_metadata = improved_splitter.create_documents_with_metadata(chunks_text)
print(f'Количество документов с метаданными: {len(chunks_with_metadata)}')
print(f'Максимальное количество символов в чанке: {max([len(chunk.page_content) for chunk in chunks_with_metadata])}')

# Выводим примеры метаданных
if chunks_with_metadata:
    print("\nПример метаданных первого чанка:")
    print(f"Название: {chunks_with_metadata[0].metadata['recipe_name']}")
    print(f"Ингредиенты: {chunks_with_metadata[0].metadata['ingredients'][:3]}...")
    print(f"Тип напитка: {chunks_with_metadata[0].metadata['drink_type']}")


Количество текстовых чанков: 72
Количество документов с метаданными: 72
Максимальное количество символов в чанке: 16710

Пример метаданных первого чанка:
Название: []{#cover.xhtml}
Ингредиенты: ['водка прозрачный лишенный запаха напиток производится методом дистилляции из картофеля или зерна', 'поясним гарнир добавляет напитку не только колорит но и новую грань вкуса и аромата например долька лайма или лимонный твист  тонкая спиралька лимонной цедры мусором называют все съедобное что добавляется к напитку в чисто эстетических целях как надетый на ободок бокала слайс лимона а китчем  самые попсовые несъедобные декоративные элементы вроде зонтика или избитых сюжетных перипетий долины кукол жаклин сьюзанн', 'большим ценителем текилы был джек керуак']...
Тип напитка: горячий напиток


### Наполним чашу `Vector Store`

In [34]:
def create_optimized_faiss_index(documents: List[Document], embeddings_model):
    """
    Создает оптимизированный FAISS индекс с IndexIVFFlat 
    для лучшей производительности поиска
    """
    logger.info("Создаем эмбеддинги для документов...")
    
    # Создаем простой индекс для получения векторов
    simple_db = FAISS.from_documents(documents, embeddings_model)
    
    # Извлекаем векторы
    vectors = simple_db.index.reconstruct_n(0, simple_db.index.ntotal)
    docstore = simple_db.docstore
    index_to_docstore_id = simple_db.index_to_docstore_id
    
    logger.info(f"Получено {len(vectors)} векторов размерности {vectors.shape[1]}")
    
    # Создаем оптимизированный IndexIVFFlat
    import faiss
    dimension = vectors.shape[1]
    n_clusters = min(50, max(4, len(vectors) // 10))  # Адаптивное количество кластеров
    
    quantizer = faiss.IndexFlatL2(dimension)
    optimized_index = faiss.IndexIVFFlat(quantizer, dimension, n_clusters)
    
    logger.info(f"Обучаем IndexIVFFlat с {n_clusters} кластерами...")
    optimized_index.train(vectors.astype('float32'))
    optimized_index.add(vectors.astype('float32'))
    
    # Оптимизируем nprobe для баланса скорости/точности
    optimized_index.nprobe = min(8, max(1, n_clusters // 3))
    logger.info(f"Установлен nprobe = {optimized_index.nprobe}")
    
    # Создаем оптимизированный FAISS объект
    optimized_db = FAISS(
        embedding_function=embeddings_model,
        index=optimized_index,
        docstore=docstore,
        index_to_docstore_id=index_to_docstore_id
    )
    
    return optimized_db

# Создание оптимизированного векторного хранилища
try:
    logger.info("Создание оптимизированного FAISS индекса...")
    db_embed = create_optimized_faiss_index(chunks_with_metadata, hf_embeddings_model)
    logger.info("✓ Оптимизированный индекс создан успешно")
except Exception as e:
    logger.error(f"Ошибка создания оптимизированного индекса: {e}")
    logger.info("Создаем стандартный индекс как fallback...")
    db_embed = FAISS.from_documents(chunks_with_metadata, hf_embeddings_model)

# Создаем директорию для сохранения
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

# Сохраняем в указанном каталоге ../data/md_vectorstore
save_path = "../data/md_vectorstore"
db_embed.index.make_direct_map()
db_embed.save_local(save_path)
logger.info(f"✓ Векторная база сохранена в: {save_path}")

# Сохраняем метаданные отдельно для анализа
metadata_file = data_dir / "recipe_metadata_md.json"
all_metadata = [doc.metadata for doc in chunks_with_metadata]

with open(metadata_file, 'w', encoding='utf-8') as f:
    json.dump(all_metadata, f, ensure_ascii=False, indent=2)
    
logger.info(f"✓ Метаданные сохранены в: {metadata_file}")


INFO:__main__:Создание оптимизированного FAISS индекса...
INFO:__main__:Создаем эмбеддинги для документов...
INFO:__main__:Получено 72 векторов размерности 768
INFO:__main__:Обучаем IndexIVFFlat с 7 кластерами...
WARNING clustering 72 points to 7 centroids: please provide at least 273 training points
INFO:__main__:Установлен nprobe = 2
INFO:__main__:✓ Оптимизированный индекс создан успешно
INFO:__main__:✓ Векторная база сохранена в: ../data/md_vectorstore
INFO:__main__:✓ Метаданные сохранены в: ../data/recipe_metadata_md.json


In [33]:
print(getattr(db_embed.index, "direct_map", None))   # None — значит не создан


<faiss.swigfaiss_avx512.DirectMap; proxy of <Swig Object of type 'faiss::DirectMap *' at 0x7c0af34672d0> >


## Нужно запустить (если используете готовое векторное хранилище)
### Испьём чашу `Vector Store`

In [20]:
# Если нужно изъять из уже сохранённого документа из базы
# Загрузка улучшенного векторного хранилища
try:
    # Пробуем загрузить из нового места
    db_embed = FAISS.load_local(
        "../data/md_vectorstore", 
        hf_embeddings_model,
        allow_dangerous_deserialization=True
    )
    logger.info("✓ Загружена улучшенная векторная база из md_vectorstore")
except Exception as e:
    # Fallback на старую базу
    logger.warning(f"Не удалось загрузить md_vectorstore: {e}")
    logger.info("Попытка загрузить оригинальную базу...")
    db_embed = FAISS.load_local(
        "../data/faiss_db_epub_mistral", 
        hf_embeddings_model,
        allow_dangerous_deserialization=True
    )
    logger.info("✓ Загружена оригинальная векторная база")


INFO:__main__:✓ Загружена улучшенная векторная база из md_vectorstore


In [28]:
# === БЛОК ТЕСТИРОВАНИЯ УЛУЧШЕННОГО ПОИСКА ===

# Тестируем поиск "Рожь и предубеждение"
test_queries = [
    "грейпфрутовый сок ржаной виски",
    "ржаной виски грейпфрут"
]

logger.info("=== ТЕСТИРОВАНИЕ УЛУЧШЕННОГО ПОИСКА ===")
for query in test_queries:
    logger.info(f"\nТестовый запрос: '{query}'")
    results = db_embed.similarity_search(query, k=3)
    
    for i, result in enumerate(results, 1):
        recipe_name = result.metadata.get('recipe_name', 'Неизвестно')
        ingredients = result.metadata.get('ingredients_str', '')
        logger.info(f"  {i}. {recipe_name}")
        logger.info(f"     Ингредиенты: {ingredients[:80]}...")
        
        # Проверяем, найден ли целевой рецепт
        if 'рож' in recipe_name.lower() and ('предуб' in recipe_name.lower() or 'гордост' in recipe_name.lower()):
            logger.info("     🎯 НАЙДЕН ЦЕЛЕВОЙ РЕЦЕПТ!")

logger.info("\n=== ТЕСТИРОВАНИЕ ЗАВЕРШЕНО ===")

INFO:__main__:=== ТЕСТИРОВАНИЕ УЛУЧШЕННОГО ПОИСКА ===
INFO:__main__:
Тестовый запрос: 'грейпфрутовый сок ржаной виски'
INFO:__main__:  1. ::: {.title1} Рип ван Дринкль :::
INFO:__main__:     Ингредиенты: рип ван винкль  всеобщий любимец деревушки в предгорьях каатскильских гор к югоз...
INFO:__main__:  2. ::: {.title1} Портрет Розовой леди :::
INFO:__main__:     Ингредиенты: 750 мл розового лимонада, свобода женщин  вот движущая сила трансатлантического ...
INFO:__main__:  3. ::: {.title1} Над пропастью с ржаным :::
INFO:__main__:     Ингредиенты: холден колфилд по сей день остается идеалом для начинающих литераторов выводящих...
INFO:__main__:
Тестовый запрос: 'ржаной виски грейпфрут'
INFO:__main__:  1. ::: {.title1} Рип ван Дринкль :::
INFO:__main__:     Ингредиенты: рип ван винкль  всеобщий любимец деревушки в предгорьях каатскильских гор к югоз...
INFO:__main__:  2. ::: {.title1} Портрет Розовой леди :::
INFO:__main__:     Ингредиенты: 750 мл розового лимонада, свобода женщин  вот 

In [29]:
# === БЛОК АНАЛИЗА И СТАТИСТИКИ ===

logger.info("\n=== СТАТИСТИКА УЛУЧШЕННОЙ ВЕКТОРНОЙ БАЗЫ ===")
logger.info(f"Общее количество документов: {len(chunks_with_metadata)}")
logger.info(f"Тип индекса: {type(db_embed.index).__name__}")
logger.info(f"Размерность векторов: {db_embed.index.d}")

# Анализ по типам напитков
drink_types = {}
for doc in chunks_with_metadata:
    dtype = doc.metadata['drink_type']
    drink_types[dtype] = drink_types.get(dtype, 0) + 1

logger.info("\nРаспределение по типам напитков:")
for dtype, count in drink_types.items():
    logger.info(f"  {dtype}: {count}")

# Анализ по количеству ингредиентов
ingredient_counts = [len(doc.metadata['ingredients']) for doc in chunks_with_metadata]
logger.info(f"\nСреднее количество ингредиентов на рецепт: {np.mean(ingredient_counts):.1f}")

# Топ-10 ингредиентов
all_ingredients = []
for doc in chunks_with_metadata:
    all_ingredients.extend(doc.metadata['ingredients'])

top_ingredients = Counter(all_ingredients).most_common(10)
logger.info("\nТоп-10 ингредиентов:")
for ingredient, count in top_ingredients:
    logger.info(f"  {ingredient}: {count}")

logger.info("\n✓ СОЗДАНИЕ УЛУЧШЕННОГО ВЕКТОРНОГО ХРАНИЛИЩА ЗАВЕРШЕНО")


INFO:__main__:
=== СТАТИСТИКА УЛУЧШЕННОЙ ВЕКТОРНОЙ БАЗЫ ===
INFO:__main__:Общее количество документов: 72
INFO:__main__:Тип индекса: IndexIVFFlat
INFO:__main__:Размерность векторов: 768
INFO:__main__:
Распределение по типам напитков:
INFO:__main__:  горячий напиток: 4
INFO:__main__:  настойка: 19
INFO:__main__:  коктейль: 21
INFO:__main__:  безалкогольный: 28
INFO:__main__:
Среднее количество ингредиентов на рецепт: 6.7
INFO:__main__:
Топ-10 ингредиентов:
INFO:__main__:  15 мл лимонного сока: 7
INFO:__main__:  50 мл водки: 5
INFO:__main__:  15 мл сока лайма: 4
INFO:__main__:  15 мл ликера триплсек: 3
INFO:__main__:  30 мл водки: 3
INFO:__main__:  15 мл персикового шнапса: 3
INFO:__main__:  60 мл водки: 3
INFO:__main__:  1 чайная ложка 5 г сахара: 3
INFO:__main__:  50 мл светлого рома: 3
INFO:__main__:  50 мл ржаного виски: 2
INFO:__main__:
✓ СОЗДАНИЕ УЛУЧШЕННОГО ВЕКТОРНОГО ХРАНИЛИЩА ЗАВЕРШЕНО


In [30]:
# Расширенное тестирование с различными запросами
test_cases = [
    {
        "name": "Тест: Рожь и предубеждение",
        "query": "ржаной виски грейпфрутовый сок",
        "expected_keywords": ["рожь", "предуб", "грейпфрут"]
    },
    {
        "name": "Тест: Джин-коктейли", 
        "query": "джин лимонный сок",
        "expected_keywords": ["джин", "лимон"]
    },
    {
        "name": "Тест: Оригинальный запрос",
        "query": "джин, лимонный сок, черносмородиновый ликер",
        "expected_keywords": ["джин", "лимон"]
    }
]

logger.info("=== РАСШИРЕННОЕ ТЕСТИРОВАНИЕ ===")
for test_case in test_cases:
    logger.info(f"\n{test_case['name']}")
    logger.info(f"Запрос: '{test_case['query']}'")
    
    results = db_embed.similarity_search(test_case['query'], k=3)
    
    for i, result in enumerate(results, 1):
        recipe_name = result.metadata.get('recipe_name', 'Неизвестно')
        logger.info(f"  {i}. {recipe_name}")
        logger.info(f"     Содержимое: {result.page_content[:100]}...")
        
        # Проверка ключевых слов
        found_keywords = [kw for kw in test_case['expected_keywords'] 
                         if kw in result.page_content.lower() or kw in recipe_name.lower()]
        if found_keywords:
            logger.info(f"     ✓ Найдены ключевые слова: {found_keywords}")

logger.info("\n=== РАСШИРЕННОЕ ТЕСТИРОВАНИЕ ЗАВЕРШЕНО ===")


INFO:__main__:=== РАСШИРЕННОЕ ТЕСТИРОВАНИЕ ===
INFO:__main__:
Тест: Рожь и предубеждение
INFO:__main__:Запрос: 'ржаной виски грейпфрутовый сок'
INFO:__main__:  1. ::: {.title1} Рип ван Дринкль :::
INFO:__main__:     Содержимое: ::: {.title1} Рип ван Дринкль :::
РИП ВАН ВИНКЛЬ (1819)
ВАШИНГТОН ИРВИНГ
Редкий случай: произведение...
INFO:__main__:  2. ::: {.title1} По вину звонит колокол :::
INFO:__main__:     Содержимое: ::: {.title1} По вину звонит колокол :::
ПО КОМ ЗВОНИТ КОЛОКОЛ (1940)
ЭРНЕСТ ХЕМИНГУЭЙ
Тут выпивка п...
INFO:__main__:  3. ::: {.title1} Над пропастью с ржаным :::
INFO:__main__:     Содержимое: ::: {.title1} Над пропастью с ржаным :::
НАД ПРОПАСТЬЮ ВО РЖИ (1951)
ДЖЕРОМ СЭЛИНДЖЕР
В своем самом ...
INFO:__main__:
Тест: Джин-коктейли
INFO:__main__:Запрос: 'джин лимонный сок'
INFO:__main__:  1. ::: {.title6} Коктейль о двух городах :::
INFO:__main__:     Содержимое: ::: {.title6} Коктейль о двух городах :::
ПОВЕСТЬ О ДВУХ ГОРОДАХ (1859)
ЧАРЛЬЗ ДИККЕНС
Диккенс на пр...
INFO:

# Месье системный промт

In [ ]:
system_prompt = """
        Ты — помощник для поиска рецептов коктейлей. У тебя есть доступ к базе данных с рецептами, вдохновлёнными литературными произведениями.
        Когда пользователь вводит ингредиенты, используй инструмент "Cocktail Recipe Finder", чтобы найти подходящие рецепты.
        Если рецепт не найден, предложи альтернативные варианты или уточни запрос.

        ===
        
        Пример:
        Пользователь: Какие коктейли можно сделать из водки и апельсинового сока?
        Агент: Использую инструмент "Cocktail Recipe Finder" для поиска рецептов. Вот что найдено: [рецепт].

        ===

        Пример корректного ответа по введённым ингредиентам (ржаной виски, грейпфрутовый сок):
        Коктейль "Рожь и предубеждение"
        Состав:
        - Ржаной виски (50 мл)
        - Грейпфрутовый сок (90 мл)
        Способ приготовления:
        - Ингредиенты выливаем в стакан рокс поверх кубиков льда, размешиваем в ритме «сердце вскачь». 
        - Хотим подчеркнуть, юные леди, мы нисколечко не предубеждены против замужества, а лишь призываем вас уяснить главное: чтобы почувствовать себя королевой, не нужно и дворца (как, впрочем, и короля).
    """

In [ ]:
# Промпт для LangChain
qa_prompt = PromptTemplate(
    template=system_prompt + "\nКонтекст: {context}\nВопрос: {question}\nОтвет:",
    input_variables=["style", "context", "question"]
)

## Полирнём `Retriver` и инициализирует `llm` для вкуса

In [ ]:
# Инициализация модели
llm = OllamaLLM(
    model="mistral:instruct",
    #temperature=0.2,          # Для креативности
    num_predict=512,         # Увеличенная длина ответа
    stop=["\\n\\n", "###"],   # Стоп-токены
    #top_k=40
)

In [ ]:
# Модифицируем параметры ретривера
retriever = db_embed.as_retriever(
    search_type="mmr",              # Maximal Marginal Relevance
    search_kwargs={
        "k": 3,                     # Уменьшаем количество возвращаемых документов
        "fetch_k": 20,              # Уменьшаем общий пул для поиска
        #"lambda_mult": 0.5          # Баланс между релевантностью и разнообразием
    }
)

# Инициализация цепочки RetrievalQA
retrieval_qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

# Миксуем рецепт коктейля `agent` 00х

In [ ]:
# Создание инструмента
agent_tools = [
    Tool(
        name="Cocktail Recipe Finder",
        func=retrieval_qa.run,
        description="Используй этот инструмент, чтобы найти рецепты коктейлей по заданным ингредиентам. Вводи список ингредиентов, и инструмент вернёт подходящие рецепты. Пример: ['сок', 'виски']."
    )
]

# создание агента
agent = initialize_agent(
    tools=agent_tools, 
    llm=llm, 
    agent="conversational-react-description",       #"zero-shot-react-description", "plan-and-execute"
    verbose=True,
    system_prompt=system_prompt                     # не забыть указать системный промт для вкуса
)

# Задаём общие параметры, после чего коктейль готов
## Вначале коктейль

In [ ]:
# Объявление функции на ввод данных от пользователя
def get_user_input():
    """
    Запрашивает у пользователя ингредиенты для поиска рецептов.
    """
    user_input = input("Введите ингредиенты, разделенные запятыми: ")
    ingredients = [ingredient.strip() for ingredient in user_input.split(",")]
    return ingredients


# Обработка ответа
def format_response(response):
    if "не знаю" in response.lower() or "не найдено" in response.lower():
        return "К сожалению, я не нашёл коктейлей с этими ингредиентами. Попробуйте другие ингредиенты."
    else:
        return response
    

# Функция выбора стиля ответа
def choose_style(styles):
    chosen_style = input("Выберите номер стиля (по умолчанию 4): ") or "4"
    return styles.get(chosen_style, styles["4"])#["name"]

In [ ]:
def main():
    # Определение стилей
    styles = {
        "1": "стиль космического ужаса и хтонического мрака Говарда Ф. Лавкрафта",
        "2": "гопническо-быдляцкий жаргон",
        "3": "экспериментальный стиль нарезок Уильяма Берроуза",
        "4": "стиль без изменений"  # Стиль по умолчанию
    }
    
    while True:
        ingredients = get_user_input()
        if not ingredients:
            print("Вы не ввели ингредиенты. Попробуйте снова.")
            continue
        
        # Преобразуем массив ингредиентов в строку
        ingredients_str = ", ".join(ingredients)
        
        # Формируем текстовый запрос
        user_prompt = (
            """Напиши, какие коктейли можно изготовить из представленных ниже ингредиентов. 
            Для каждого коктейля опиши подробный способ приготовления и укажи пропорции. 
            Отвечай строго на русском языке, используя чёткий и лаконичный формат. 
            Не добавляй лишнюю информацию, которая не относится к рецепту. 
            Ингредиенты: {ingredients}""".format(ingredients=ingredients_str)
        )
        response = retrieval_qa.invoke({"query": user_prompt})  
        
        # Получаем ответ без вывода его на экран
        original_response = response["result"]
        # Выбор стиля
        style = choose_style(styles)
        prompt = PromptTemplate(input_variables=['output_text', 'style'],
                                template='''Добро пожаловать в поиск коктейлей!\nПерепиши этот текст в заданном стиле: {output_text}\nСтиль: {style}.\nРезультат:\n''')
        
        # Генерация текста с использованием оригинального ответа и выбранного стиля
        styled_prompt = prompt.format(output_text=original_response, style=style)
        
        # Применение выбранного стиля к оригинальному ответу
        styled_response = llm.invoke(styled_prompt)
        print(styled_response)
        break
    
    return styled_response

if __name__ == "__main__":
    # Сохраняем результат в переменную
    recipe = main()